<a href="https://colab.research.google.com/github/jcesarmph-ui/SmartPredict/blob/main/M4_Otimizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importando bibliotecas

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [2]:
# Carregando dataset

df = pd.read_csv('dados_trata.csv')

df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,False,False,True
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,False,True,False
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,False,True,False
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,False,True,False
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,False,True,False


In [3]:
# Variáveis de entrada

X = df.drop('Machine failure', axis=1)

# Variável alvo

y = df['Machine failure']

In [4]:
# Divisão treino e teste

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
# Hiperparâmetros

parametros = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

In [6]:
# Criando modelo

modelo = RandomForestClassifier()

# Otimização

random_search = RandomizedSearchCV(
    modelo,
    parametros,
    n_iter=5,
    cv=5,
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_iter=5,
                   param_distributions={'max_depth': [5, 10, 20],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42)

In [7]:
# Melhor modelo

melhor_modelo = random_search.best_estimator_

print(melhor_modelo)

RandomForestClassifier(max_depth=5, min_samples_split=10, n_estimators=200)


Foi utilizada a técnica RandomizedSearchCV para otimizar automaticamente os hiperparâmetros do modelo Random Forest, buscando melhorar sua performance e capacidade de generalização.

#Interpretação dos Parâmetros
-O modelo utilizou 200 árvores de decisão, aumentando a robustez e a capacidade de identificar padrões complexos nos dados.
-As árvores foram limitadas a uma profundidade máxima de 5 níveis, reduzindo o risco de overfitting e tornando o modelo mais generalizável.
-Uma divisão em um nó da árvore só ocorre quando existem pelo menos 10 amostras disponíveis, evitando decisões excessivamente específicas causadas por poucos dados.

A utilização desses parâmetros permitiu que o modelo alcançasse elevado desempenho, mantendo estabilidade e reduzindo riscos de superajuste durante o treinamento.

In [8]:
# Fazendo previsões

y_pred = melhor_modelo.predict(X_test)

In [9]:
# Resultado final

acuracia = accuracy_score(y_test, y_pred)

print("Acurácia:", acuracia)

print("\nRelatório:")

print(classification_report(y_test, y_pred))

Acurácia: 0.999

Relatório:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1939
           1       1.00      0.97      0.98        61

    accuracy                           1.00      2000
   macro avg       1.00      0.98      0.99      2000
weighted avg       1.00      1.00      1.00      2000



In [10]:
# Cross Validation

scores = cross_val_score(
    melhor_modelo,
    X,
    y,
    cv=5
)

print("Scores:", scores)

print("Média:", scores.mean())

Scores: [0.9995 0.999  0.6715 0.998  0.999 ]
Média: 0.9334


A validação cruzada foi utilizada para avaliar a capacidade de generalização do modelo em diferentes divisões do conjunto de dados. Essa técnica reduz o risco de overfitting, garantindo que o desempenho do modelo não esteja limitado apenas a uma única separação entre treino e teste.

Os resultados obtidos foram:

Scores: [0.9995, 0.9990, 0.6715, 0.9980, 0.9990]
Média: 0.9334

A média de desempenho de aproximadamente 93,34% indica que o modelo apresentou alta capacidade preditiva de forma geral. A maioria das divisões obteve resultados extremamente elevados, próximos de 100% de acurácia.

Entretanto, uma das dobras apresentou desempenho inferior (0.6715), indicando que existem variações na distribuição dos dados entre algumas partes do dataset. Isso sugere que o modelo pode estar sensível a determinados padrões específicos presentes em algumas divisões dos dados.

Mesmo com essa variação, o resultado médio demonstra que o modelo possui boa robustez e capacidade de generalização, sendo adequado para aplicações de manutenção preditiva industrial.